# ?? Microsoft Foundry (.NET) ?????????????? RAG

## ?? ????

???????????Microsoft Foundry ? .NET ? Microsoft Agent Framework ????????????????? Retrieval-Augmented Generation (RAG) ???????????????????????????????????????????????????????????????????????????????????????????????????????

**???????????? RAG ???:**
- ?? **??????????????**: Azure AI ????????????????
- ?? **?????????**: ???????????????????????
- ??? **????????**: ???????????????????????
- ?? **??????????????**: ??????????? RAG ????

## ?? ???????? RAG ???????

### ?????????????????
- **Microsoft Foundry**: ????????????????????????????????? AI ????????
- **Persistent Agents**: ???????????????????????????
- **Vector Store Management**: ???????????????????????????
- **Identity Integration**: Azure AD ???????????????

### .NET ?????????????
- **????**: RAG ?????????????????
- **??????????**: ????????????????????????
- **?????**: ?????????????????????????????
- **??????**: ?????????????? Azure ??????

## ??? ?????????

### Enterprise RAG ??????
```csharp
Document Upload ? Security Validation ? Vector Processing ? Index Creation
                      ?                    ?                  ?
User Query ? Authentication ? Semantic Search ? Context Ranking ? AI Response
```

### ?? .NET ???????
- **Azure.AI.Agents.Persistent**: ????????????????????????????
- **Azure.Identity**: ??? Azure ????????????????
- **Microsoft.Agents.AI.AzureAI**: Azure ??????????????????????
- **System.Linq.Async**: ??????? LINQ ??

## ?? ?????????????

### ???????????????
- **Azure AD ??**: ??????????????????????
- **??????????**: ???????????????????????
- **?????**: ??????????????????????
- **????**: ???????????????????????????

### ????????????????
- **???????**: Azure ?????????????
- **?????**: ????????????????????????
- **???????**: ??????????????????????????????????
- **?????????**: ????????????

### ?????
- **???????**: RAG ??????????????????
- **?????????**: ??????????????????
- **?????????**: ????????????????????
- **????**: ????????????

## ?? ???????????

**????:**
- .NET 9.0 SDK ??
- Visual Studio 2022 ??? C# ???????? VS Code
- AI Foundry ??????? Azure ?????????

**?? NuGet ?????:**
```xml
<PackageReference Include="Microsoft.Extensions.AI" Version="9.9.0" />
<PackageReference Include="Azure.AI.Agents.Persistent" Version="1.2.0-beta.5" />
<PackageReference Include="Azure.Identity" Version="1.15.0" />
<PackageReference Include="System.Linq.Async" Version="6.0.3" />
<PackageReference Include="DotNetEnv" Version="3.1.1" />
```

**Azure ?????????:**
```bash
# Azure CLI ?????????????
az login
az account set --subscription "your-subscription-id"
```

**???? (.env ????):**
```env
# Microsoft Foundry configuration (automatically handled via Azure CLI)
# Ensure you're authenticated to the correct Azure subscription
```

## ?? ???????? RAG ????

### ????????????
- **????????**: ?????????????????????
- **??????????**: ??????????????????
- **???????**: ???????????????????
- **???????**: ????????????????

### ???????????
- **????????**: ????????????????????????????????
- **???????**: ??????????????
- **?????**: ???????????????????????????????
- **???????**: ?????????????????????

### ??????????
- **????????????????**: ??????????????????
- **?????**: ???????????????
- **??????**: ???? RAG ???????????
- **????????**: PII ??????????

## ?? ??????????????????

### ?????
```csharp
// Azure AD integrated authentication
var credential = new AzureCliCredential();
var agentsClient = new PersistentAgentsClient(endpoint, credential);

// Role-based access validation
if (!await ValidateUserPermissions(user, documentId))
{
    throw new UnauthorizedAccessException("Insufficient permissions");
}
```

### ?????
- **???**: ???????????????????????????
- **??????**: ???????????????? Azure AD ??
- **??????**: ??????????????????????
- **?????????**: ???????????????????

## ?? ??????????

### ?????????
```csharp
// Efficient async document processing
await foreach (var document in documentStream.AsAsyncEnumerable())
{
    await ProcessDocumentAsync(document, cancellationToken);
}
```

### ?????
- **?????????**: ????????????????????
- **?????????**: ?????????????????
- **??????????**: ?????????????????
- **????**: ??? Azure ????????????????

### ???????
- **????????**: ????????????????
- **???????????**: ????????????????????
- **???????????**: ?????????????????????
- **???????**: ??????????????????????

## ?? ??????????????

### ??????????
- **?????**: ??????????????????????????
- **?????**: ????????????????????
- **????????**: ????????????????
- **????????**: ????????????????

### ?????????
- **???????????**: ?????????????????
- **????????**: ????????????????
- **??????????????**: ????????????????
- **FAQ ????**: ?????????????????? FAQ ??

### ??????????
- **??????**: ?????????????????
- **??????????**: ????????????????????
- **?????**: ????????????????????
- **????**: ???????????????????????

## ?? ??????????

### ??????????
- **Application Insights**: ??????????????????
- **?????????**: ??????? KPI ???????
- **????????**: ?????????????????????
- **??????????**: ?????????????????????????

### ????????????
- **?????????**: ????????????????????????
- **????**: ???????????????????????
- **??????**: ????????????????????????
- **????**: ?????????????????

??????????????????????????? RAG ???????????????????????????????????????????????????????! ?????


In [ ]:
#r "nuget: Microsoft.Extensions.AI, 9.9.1"

In [ ]:
#r "nuget: Azure.AI.Agents.Persistent, 1.2.0-beta.5"
#r "nuget: Azure.Identity, 1.15.0"
#r "nuget: System.Linq.Async, 6.0.3"

In [ ]:
#r "nuget: Microsoft.Agents.AI.AzureAI, 1.0.0-preview.251001.3"

In [ ]:
#r "nuget: Microsoft.Agents.AI, 1.0.0-preview.251001.3"

In [ ]:
#r "nuget: DotNetEnv, 3.1.1"

In [ ]:
using System;
using System.Linq;
using Azure.AI.Agents.Persistent;
using Azure.Identity;
using Microsoft.Agents.AI;

In [ ]:
 using DotNetEnv;

In [ ]:
Env.Load("../../../.env");

In [ ]:
var azure_foundry_endpoint = Environment.GetEnvironmentVariable("AZURE_AI_PROJECT_ENDPOINT") ?? throw new InvalidOperationException("AZURE_AI_PROJECT_ENDPOINT is not set.");
var azure_foundry_model_id = Environment.GetEnvironmentVariable("AZURE_AI_MODEL_DEPLOYMENT_NAME") ?? "gpt-4.1-mini";

In [ ]:
string pdfPath = "./document.md";

In [ ]:
using System.IO;

async Task<Stream> OpenImageStreamAsync(string path)
{
	return await Task.Run(() => File.OpenRead(path));
}

var pdfStream = await OpenImageStreamAsync(pdfPath);

In [ ]:
var persistentAgentsClient = new PersistentAgentsClient(azure_foundry_endpoint, new AzureCliCredential());

In [ ]:
PersistentAgentFileInfo fileInfo = await persistentAgentsClient.Files.UploadFileAsync(pdfStream, PersistentAgentFilePurpose.Agents, "demo.md");

In [ ]:
PersistentAgentsVectorStore fileStore =
            await persistentAgentsClient.VectorStores.CreateVectorStoreAsync(
                [fileInfo.Id],
                metadata: new Dictionary<string, string>() { { "agentkey", bool.TrueString } });

In [ ]:
PersistentAgent agentModel = await persistentAgentsClient.Administration.CreateAgentAsync(
            azure_foundry_model_id,
            name: "DotNetRAGAgent",
            tools: [new FileSearchToolDefinition()],
            instructions: """
                You are an AI assistant designed to answer user questions using only the information retrieved from the provided document(s).

                - If a user's question cannot be answered using the retrieved context, **you must clearly respond**: 
                "I'm sorry, but the uploaded document does not contain the necessary information to answer that question."
                - Do not answer from general knowledge or reasoning. Do not make assumptions or generate hypothetical explanations.
                - Do not provide definitions, tutorials, or commentary that is not explicitly grounded in the content of the uploaded file(s).
                - If a user asks a question like "What is a Neural Network?", and this is not discussed in the uploaded document, respond as instructed above.
                - For questions that do have relevant content in the document (e.g., Contoso's travel insurance coverage), respond accurately, and cite the document explicitly.

                You must behave as if you have no external knowledge beyond what is retrieved from the uploaded document.
                """,
            toolResources: new()
            {
                FileSearch = new()
                {
                    VectorStoreIds = { fileStore.Id },
                }
            },
            metadata: new Dictionary<string, string>() { { "agentkey", bool.TrueString } });

In [ ]:
AIAgent agent = await persistentAgentsClient.GetAIAgentAsync(agentModel.Id);

In [ ]:
AgentThread thread = agent.GetNewThread();

In [ ]:
Console.WriteLine(await agent.RunAsync("Can you explain Contoso's travel insurance coverage?", thread));

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
????:
???AI???????Co-op Translator?(https://github.com/Azure/co-op-translator)?????????????????????????????????????????????????????????????????(??????????)???(??????)???????????????????????????????????????????????????????????????????????????
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
